Working on pdf with Spark

In [0]:
%python
!pip install pypdf
!pip install langchain-text-splitters
dbutils.library.restartPython()

In [0]:
%python
SOURCE_PATH = '/Volumes/ai2605/ai/unstructured/pdfs/'
raw_files_pdf = (
  spark
    .read
    .format('binaryFile')
    .option('pathGlobFilter', '*.pdf') 
    .option('recursiveFileLookup', 'true')
    .load(SOURCE_PATH)
)
# DBTITLE 1,Display raw files
display(raw_files_pdf.count())


In [0]:
%python
import os
import pyspark.sql.functions as F
import pyspark.sql.types as T
import io
import pypdf

def parse_byte_json(   
    raw_doc_content_byte: bytes,
    document_path: str,
    content_key: str
) -> dict:
    """
    This function takes a byte array and returns a dictionary with the following keys:
    - 'path': the path of the document
    - 'modificationTime': the modification time of the document
    - 'length': the length of the document
    - 'content': the content of the document
    """
    pdf_file = io.BytesIO(raw_doc_content_byte)
    pdf_reader = pypdf.PdfReader(pdf_file)
    
    # Extract text from all pages
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()
    
    return {
        'path': document_path,
        'content': text
    }
    

from functools import partial

parser_udf = F.udf(
    partial(
        parse_byte_json,
        content_key='pdf_content'
    ),
    returnType=T.StructType([
        T.StructField('path', T.StringType(), True),
        T.StructField('content', T.StringType(), True)
    ])
)

parsed_files_staging_df = raw_files_pdf.withColumn(
    "parsed",
    parser_udf(F.col("content"), F.col("path"))
)
#display(parsed_files_staging_df)
(parsed_files_staging_df.write
 .mode('overwrite')
 .option("overwriteSchema", "true")
 .saveAsTable('tbl_parsed_files')
)

In [0]:
select * from tbl_parsed_files desc 

In [0]:
%python
%pip install langchain-text-splitters

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 256

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.functions import col, explode, array, lit, struct


def chunk_parsed_contents(
    parsed_contents: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP
)->dict:
    """
    This function takes a dataframe with parsed contents and returns a dataframe with chunked contents.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )
    chunks = text_splitter.split_text(parsed_contents)
    return {
        'chunked_text': chunks
    }

chunker_udf = F.udf(
    partial(    
        chunk_parsed_contents,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    ),
    returnType=T.StructType([
        T.StructField('chunked_text', T.ArrayType(T.StringType()), True)
    ])
) 

chunked_files_staging_df = parsed_files_staging_df.withColumn(
    "chunks",
    chunker_udf(F.col("parsed.content"))
)
chunked_files_df = chunked_files_staging_df.select(
    'path',
    'modificationTime',
    'length',
    F.explode(col('chunks.chunked_text')).alias('chunked_text'),
    F.md5(col('chunked_text')).alias('chunked_id') 
)

# DBTITLE 1,Display chunked files
display(chunked_files_df)
(chunked_files_df.write
 .mode('overwrite')
 .option("overwriteSchema", "true")
 .saveAsTable('tbl_chunked_files')
)
       


In [0]:
ALTER TABLE workspace.default.tbl_chunked_files 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
ALTER TABLE workspace.default.tbl_chunked_files 
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 30 days');

In [0]:
%python
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

for endpoint in w.serving_endpoints.list():
    print(endpoint.name)
     

In [0]:
%python
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
%python
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

client.list_endpoints()

In [0]:
%python
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

# Get or create Delta Sync Index with managed embeddings
try:
    index = client.create_delta_sync_index(
        endpoint_name="20260718_aisearch",
        source_table_name="workspace.default.tbl_chunked_files",
        index_name="workspace.default.tbl_chunked_files_index",
        pipeline_type="TRIGGERED",
        primary_key="chunked_id",
        embedding_source_column="chunked_text",
        embedding_model_endpoint_name="databricks-qwen3-embedding-0-6b"
    )
    print("✓ Vector Search index created successfully!")
except Exception as e:
    if "already exists" in str(e):
        index = client.get_index(
            endpoint_name="20260718_aisearch",
            index_name="workspace.default.tbl_chunked_files_index"
        )
        print("✓ Vector Search index already exists, retrieved successfully!")
    else:
        raise

print(f"Index name: {index.name}")
print(f"Index status: {index.describe()['status']['ready']}")
print("\nYou can sync the index by calling: index.sync()")